# mini-sglang `shm` / `hssm` 通信后端实验

本 notebook 对齐 `mini-sglang-hssm通信后端优化方案.md` 的测试方法：先验证协议和槽位，再测 host copy，最后在固定模型、输入输出长度、page size、chunk 和并发度下比较 `none`、`shm`、`hssm`。

端到端部分要求 Linux + CUDA + 两张 GPU；CPU 环境可运行前两部分。每个后端会单独创建和清理 runtime。

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'minisgl').exists():
    ROOT = ROOT.parent
if not (ROOT / 'minisgl').exists():
    raise FileNotFoundError('请从 mini-sglang 目录或其子目录启动 Jupyter')
sys.path.insert(0, str(ROOT))

from examples.hssm_experiment import (
    benchmark_backend, benchmark_copy, to_rows, verify_protocol, verify_ring,
)


In [ ]:
# 方案 9.1：协议、TransferKey/Meta 版本校验、槽位回绕和边界检查
print(verify_protocol())
print(verify_ring(payload_size=4096, slots=3))


In [ ]:
# 方案 8/9：host shared-memory copy 微基准。单位 GiB/s 是十进制近似。
copy_rows = benchmark_copy(warmup=3, repeat=20, page_locked_host_memory=False)
copy_rows


In [ ]:
# 仅 CUDA 环境执行：测试 cudaHostRegister 后的 host ring。
import torch
if torch.cuda.is_available():
    page_locked_host_memory_copy_rows = benchmark_copy(warmup=3, repeat=20, page_locked_host_memory=True)
    page_locked_host_memory_copy_rows
else:
    print('跳过：当前环境没有 CUDA')


## 双卡端到端实验
将 `MODEL_PATH` 改为本地 mini-sglang 支持的模型目录。固定 `input_len/output_len/batch_size/page_size/chunk_tokens` 后，分别运行 `none`、`shm`、`hssm`。`none` 是单进程基线，`shm` 是未注册 page-locked host memory 的 SHM 路径，`hssm` 是优化候选。

In [ ]:
MODEL_PATH = '/path/to/local/model'
DTYPE = 'auto'  # 可选：auto、float16、bfloat16、float32
DEVICES = (0, 1)
INPUT_LEN = 128
OUTPUT_LEN = 32
BATCH_SIZE = 1
WARMUP = 1
REPEAT = 5
PAGE_SIZE = 16
CHUNK_TOKENS = 512
SLOT_COUNT = 4

assert Path(MODEL_PATH).exists(), '请先设置 MODEL_PATH'
import torch
assert torch.cuda.is_available() and torch.cuda.device_count() >= 2, '端到端实验需要两张 CUDA GPU'


In [ ]:
results = []
for backend in ('none', 'shm', 'hssm'):
    print(f'Running {backend} ...')
    results.append(benchmark_backend(
        MODEL_PATH, backend, input_len=INPUT_LEN, output_len=OUTPUT_LEN,
        batch_size=BATCH_SIZE, warmup=WARMUP, repeat=REPEAT,
        page_size=PAGE_SIZE, chunk_tokens=CHUNK_TOKENS,
        slot_count=SLOT_COUNT, devices=DEVICES, dtype=DTYPE,
    ))

rows = to_rows(results)
try:
    import pandas as pd
    display(pd.DataFrame(rows))
except ImportError:
    rows


In [ ]:
# 相对 none 基线的变化；正值表示提升，负值表示下降。
baseline = next(row for row in rows if row['backend'] == 'none')
comparison = []
for row in rows:
    comparison.append({
        'backend': row['backend'],
        'ttft_p90_delta_pct': 100 * (baseline['ttft_p90_ms'] - row['ttft_p90_ms']) / baseline['ttft_p90_ms'],
        'tpot_p90_delta_pct': 100 * (baseline['tpot_p90_ms'] - row['tpot_p90_ms']) / baseline['tpot_p90_ms'],
        'throughput_delta_pct': 100 * (row['output_tok_per_s'] - baseline['output_tok_per_s']) / baseline['output_tok_per_s'],
        'success_rate': row['success_rate'],
    })
try:
    display(pd.DataFrame(comparison))
except NameError:
    comparison


## 结果解释
- `TTFT P90/P99`：从 `add_request` 后到该请求第一次输出的墙钟时间。
- `TPOT P90/P99`：从首 token 到结束时间除以其余输出 token 数；`output_len=1` 时该值只用于参考。
- `output_tok_per_s`：成功请求生成 token 数除以各请求总耗时之和，适合观察后端在固定工作量下的相对变化。
- `success_rate` 和 `transfer_failures`：对应方案中的失败率/异常回收验收。
- 该 notebook 的 `hssm` 只验证当前 mini-sglang 中的 `cudaHostRegister` 路径；它不会把原 SGLang PR 的性能数字当作本机承诺。正式报告应记录 GPU 型号、PCIe 拓扑、驱动、CUDA、torch、模型、dtype、page size、chunk、slot、并发度和每轮重启情况。